# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Follows `checklist.md`.

Data notes: no date column, so there is no time ordering to respect; `device_serial` is unique per row (7000 rows, 7000 serials), so there are no repeated entities. A stratified random split is therefore appropriate. `device_serial` is an identifier and is dropped. The remaining columns (channel, device age, price, reported daily hours) are all recorded at sale/registration, i.e. known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

df = pd.read_csv('data.csv')
print(df.shape, 'unique serials:', df.device_serial.nunique(), 'missing:', int(df.isna().sum().sum()))
print('claim rate:', df.claim_within_year.mean())
df.describe()

## 1. Hold out the test set (before any modelling)
Serials are unique and there is no date, so a stratified random 20% split. The test set is not touched until step 6.

In [ ]:
TARGET='claim_within_year'
FEATURES=['channel','device_age_years','price_eur','daily_hours']  # identifier device_serial dropped
train, test = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=0)
X_tr, y_tr = train[FEATURES], train[TARGET]
X_te, y_te = test[FEATURES], test[TARGET]
print(len(train), len(test), y_tr.mean(), y_te.mean())

## 2-4. Candidate models (named before scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: logistic regression (one-hot channel, scaled numerics), C in {0.1, 1, 10}.
3. `hgb`: histogram gradient boosting, small grid over learning rate / depth.

Cross-validation: 5-fold stratified on training rows only (entities are unique, so rows = entities). All preprocessing sits inside the Pipeline, so it is fit within each fold. Selection metric: ROC AUC (threshold-free); the decision threshold is then picked on out-of-fold predictions to maximise F1.

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
def prep(): return ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
candidates = {'baseline': Pipeline([('m',DummyClassifier(strategy='prior'))])}
for C_ in [0.1,1,10]:
    candidates[f'logreg_C{C_}'] = Pipeline([('p',prep()),('m',LogisticRegression(C=C_,max_iter=1000))])
for lr in [0.03,0.1]:
    for d in [2,3,None]:
        candidates[f'hgb_lr{lr}_d{d}'] = Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(learning_rate=lr,max_depth=d,max_iter=200,random_state=0))])
cv = StratifiedKFold(5, shuffle=True, random_state=1)
oof, rows = {}, []
for name, pipe in candidates.items():
    p = cross_val_predict(pipe, X_tr, y_tr, cv=cv, method='predict_proba')[:,1]
    oof[name]=p
    rows.append((name, roc_auc_score(y_tr,p), average_precision_score(y_tr,p)))
cvres = pd.DataFrame(rows, columns=['model','cv_auc','cv_ap']).sort_values('cv_auc',ascending=False)
cvres

## 5. Choose model and threshold on CV only

In [ ]:
best = cvres.iloc[0].model
p = oof[best]
ths = np.linspace(0.02,0.6,59)
f1s = [f1_score(y_tr, p>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr,3), '| CV F1:', round(max(f1s),3))
print('CV precision/recall at threshold:', precision_score(y_tr,p>=thr), recall_score(y_tr,p>=thr))

**Choice (recorded before test scoring):** the model above (highest CV AUC among the named candidates) with the F1-maximising out-of-fold threshold printed above. No further changes will be made after seeing test results.

## 6. Refit on all training rows; score the test set once

In [ ]:
final = candidates[best].fit(X_tr, y_tr)
pt = final.predict_proba(X_te)[:,1]
base = candidates['baseline'].fit(X_tr,y_tr).predict_proba(X_te)[:,1]
pred = pt>=thr
print(f'TEST ROC AUC : {roc_auc_score(y_te,pt):.3f}  (baseline {roc_auc_score(y_te,base):.3f})')
print(f'TEST PR AUC  : {average_precision_score(y_te,pt):.3f}  (prevalence {y_te.mean():.3f})')
print(f'TEST precision/recall/F1 at thr {thr:.2f}: {precision_score(y_te,pred):.3f} / {recall_score(y_te,pred):.3f} / {f1_score(y_te,pred):.3f}')
rng=np.random.default_rng(0); aucs=[]
for _ in range(1000):
    i=rng.integers(0,len(y_te),len(y_te)); yy=y_te.values[i]
    if yy.min()!=yy.max(): aucs.append(roc_auc_score(yy,pt[i]))
print('AUC 95% bootstrap CI:', np.percentile(aucs,[2.5,97.5]).round(3))